# INF-473 - Introducción a la Inteligencia Artificial Explicable
## 1er Semestre 2026 -- Prof. Raquel Pezoa
<center>
    <h1> Regresión probabilística con varianza no constante </h2>
</center>

* En la clase anterior hicimos el siguiente supuesto:

$$y_i \mid x_i \sim \mathcal{N}(\mu(x_i), \sigma^2)$$

donde:

$$\mu(x_i) = ax_i + b$$

y $\sigma$ era constante.

* Es decir, usamos un **modelo lineal**, donde suponíamos el mismo nivel de ruido para todos los valores de $x$.

-Ahora veremos una extensión natural:


$$y_i \mid x_i \sim \mathcal{N}(\mu(x_i), \sigma(x_i)^2)$$

* Ahora el modelo aprende dos funciones:


$$\mu(x)$$


y

$$\sigma(x)$$

* En ambos casos, se minimiza la NLL, pero cambiará como modelamos $\mu(x)$ y $\sigma(x)$

### Antes:
$$\mu(x) = ax +b $$

### Ahora:
$$\mu(x) = f_{\theta}(x) \,\,\,\,\,\,\,\,\,\, \sigma(x)=g_{\theta}(x) $$

* Ahora usamos una red neuronal para modelar estas funciones no lineales.
* En el caso que $\sigma$ no es constante, no equivale a MSE.
* Notar que todavía no es enfoque probabilístico Bayesiano, porque los parámetros de la red siguen siendo valores fijos.


## Datos con ruido no constante

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n = 150
x = np.linspace(-3, 3, n).reshape(-1, 1)

# Media verdadera
mu_true = 2 * x - 1

# Ruido que depende de x
sigma_true = 0.2 + 0.3 * np.abs(x)

y = mu_true + np.random.normal(0, sigma_true)

plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.6, label="Datos")
plt.plot(x, mu_true, color="black", label="Media verdadera")
plt.fill_between(
    x.flatten(),
    (mu_true - 2 * sigma_true).flatten(),
    (mu_true + 2 * sigma_true).flatten(),
    alpha=0.2,
    label="Ruido verdadero aprox."
)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Datos con ruido dependiente de x")
plt.show()

* En estos datos, la varianza no es constante.

* Cerca de $x=0$, los datos tienen menos ruido.

* Para valores grandes de $|x|$, los datos tienen más dispersión.

* Por lo tanto, un modelo con $\sigma$ constante no puede representar bien este comportamiento.

## Modelo con dos salidas

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(32, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(2)
])

El modelo entrega dos valores para cada entrada $x$:

$$[\mu(x), \sigma(x)]$$

donde $\sigma(x)$ será transformado para obtener una desviación estándar positiva.

Usaremos:

$$\sigma(x) = \text{softplus}(\sigma(x)) + \epsilon$$

La función `softplus` asegura que $\sigma(x)$ sea positiva.

### NLL heterocedástica


Esta función define la **negative log-likelihood (NLL)** para un modelo donde:

$$
y \mid x \sim \mathcal{N}(\mu(x), \sigma(x)^2)
$$

La red neuronal predice dos valores para cada entrada:
- $\mu(x)$  la media  
- $\sigma(x)$  la desviación estándar  


Para una distribución normal, se tiene:

$$
-\log p(y_i \mid x_i)
=
\frac{1}{2}\log\left(2\pi\sigma(x_i)^2\right)
+
\frac{1}{2}
\left(
\frac{y_i - \mu(x_i)}{\sigma(x_i)}
\right)^2
$$

Este término tiene dos componentes:

#### 1. Penalización por incertidumbre

$$
\frac{1}{2}\log\left(\sigma(x)^2\right)
$$

Este término penaliza valores grandes de $$\sigma(x)$$.

#### 2. Penalización por error

$$
\frac{1}{2}
\left(
\frac{y - \mu(x)}{\sigma(x)}
\right)^2
$$

Este término penaliza la diferencia entre la predicción y el valor real.

Finalmente, se minimiza:

$$
\frac{1}{n} \sum_{i=1}^{n} -\log p(y_i \mid x_i)
$$

In [ ]:
def heteroscedastic_nll(y_true, y_pred):
    mu = y_pred[:, 0:1] # primer columna entrega la media
    raw_sigma = y_pred[:, 1:2] # segunda columna se asocia a sigma
    
    sigma = tf.nn.softplus(raw_sigma) + 1e-6 # aplicamos softmax para valores positivos
    
    nll = 0.5 * tf.math.log(2 * np.pi * sigma**2) + 0.5 * ((y_true - mu) / sigma)**2
    
    return tf.reduce_mean(nll)

## Training

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=heteroscedastic_nll
)

history = model.fit(
    x, y,
    epochs=1000,
    verbose=0
)

## Predicciones

In [ ]:
pred = model.predict(x)

mu_pred = pred[:, 0:1]
sigma_pred = tf.nn.softplus(pred[:, 1:2]).numpy() + 1e-6

plt.figure(figsize=(8, 5))

plt.scatter(x, y, alpha=0.5, label="Datos")
plt.plot(x, mu_true, color="black", linestyle="--", label="Media verdadera")
plt.plot(x, mu_pred, label="Media predicha")

plt.fill_between(
    x.flatten(),
    (mu_pred - 2 * sigma_pred).flatten(),
    (mu_pred + 2 * sigma_pred).flatten(),
    alpha=0.25,
    label="Intervalo aproximado ±2σ(x)"
)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Regresión probabilística con σ(x)")
plt.legend()
plt.show()

## Comentarios

Este modelo aprende una distribución condicional:

$$p(y \mid x)$$

pero ahora con varianza dependiente de $x$:

$$ y \mid x \sim \mathcal{N}(\mu(x), \sigma(x)^2)$$

Esto permite capturar incertidumbre aleatoria que cambia según la región del espacio de entrada.

A esta incertidumbre se le suele llamar: 
* aleatoric uncertainty, o 
* incertidumbre aleatoria, o
* incertidumbre de los datos.

## ¿Esto ya es Bayesiano?

**No.**

* Aunque el modelo aprende una varianza $\sigma(x)$, los parámetros de la red siguen siendo valores fijos.

* Es decir, después del entrenamiento tenemos un único conjunto de pesos.

* Por lo tanto, el modelo puede expresar:

$$\text{incertidumbre en } y \mid x$$

* pero no expresa:

$$\text{incertidumbre sobre los parámetros del modelo}$$

* En el enfoque Bayesiano, el siguiente paso será tratar los parámetros como variables aleatorias.

* En vez de aprender una única recta o una única red, aprenderemos una distribución sobre posibles modelos.

<div style="background-color: #d1ecf1; padding: 10px; border-left: 5px solid #0c5460; border-radius: 5px;">

* La idea importante es que podemos hacer el modelo probabilístico más flexible, permitiendo que la varianza dependa de $x$. 

* Sin embargo, esto sigue siendo no Bayesiano: la incertidumbre está en los datos, no en los parámetros. 

</div>